# 통합 벤치마크 v2 (P0 반영)

이 노트북은 벤치마크 구현을 복사하지 않고 저장소의 `code/test/run_benchmark_v2.py`를 직접 실행합니다. 따라서 source-paper 단위 Recall/NDCG, Paper-only negative sampling, train citation 후보 제외가 로컬 코드와 동일하게 적용됩니다. 실행 전 P0 변경이 포함된 커밋을 원격 브랜치에 push하세요.

In [ ]:
import torch
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

In [ ]:
import os, subprocess, sys
REPOSITORY = 'https://github.com/YHFTF/arxiv-conversational-recommender.git' # @param {type:'string'}
BRANCH = 'main' # @param {type:'string'}
PROJECT_ROOT = '/content/arxiv-recsys'
if os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_ROOT, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPOSITORY, PROJECT_ROOT], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')
DATA_ARCHIVE = '/content/drive/MyDrive/colab_data.zip' # @param {type:'string'}
SHARED_OUTPUT = '/content/drive/MyDrive/arxiv-recsys-output' # @param {type:'string'}
assert os.path.exists(DATA_ARCHIVE), f'데이터 압축 파일이 없습니다: {DATA_ARCHIVE}'
subprocess.run(['unzip', '-q', '-o', DATA_ARCHIVE, '-d', PROJECT_ROOT], check=True)
required = ['subdataset/build_hetero_graph_v2.pt', 'subdataset/arxiv_master_final.json', 'output/knowledge_meta.json']
missing = [path for path in required if not os.path.exists(os.path.join(PROJECT_ROOT, path))]
assert not missing, f'필수 데이터가 없습니다: {missing}'
os.makedirs(SHARED_OUTPUT, exist_ok=True)
shutil.copytree(os.path.join(PROJECT_ROOT, 'output'), SHARED_OUTPUT, dirs_exist_ok=True)
print('데이터 준비 완료 · 결과 저장:', SHARED_OUTPUT)

In [ ]:
# 빈 값이면 다섯 모델을 모두 실행합니다. 예: 'LightGCN + Knowledge (Ours)'
ONLY_MODEL = '' # @param {type:'string'}
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['OUTPUT_DIR'] = SHARED_OUTPUT
command = [sys.executable, 'code/test/run_benchmark_v2.py']
if ONLY_MODEL.strip():
    command.extend(['--only', ONLY_MODEL.strip()])
print('실행:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)

In [ ]:
from pathlib import Path
results = sorted(Path(SHARED_OUTPUT, 'benchmark').glob('benchmark_v2_results_*.json'))
assert results, '결과 JSON이 생성되지 않았습니다.'
print('최신 결과:', results[-1])
print(results[-1].read_text(encoding='utf-8'))